[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sprice134/DualSight/blob/master/dataCollection.ipynb)


# Installing Imports

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install torch torchvision torchaudio
    !pip install git+https://github.com/facebookresearch/segment-anything.git
    !pip install opencv-python pycocotools matplotlib ipykernel
    !pip install ultralytics
    !mkdir -p models
    !wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth -P models/
    !git clone https://github.com/sprice134/DualSight.git

In [9]:
import os
import cv2
import torch
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from shapely.geometry import Polygon, Point
from skimage.draw import polygon as skpolygon
from skimage.measure import regionprops
from sklearn.metrics import r2_score
from skimage.draw import polygon

# Import SAM modules
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

print("CUDA available?", torch.cuda.is_available())



CUDA available? True


In [6]:
def get_jpg_files(directory):
    jpg_files = []
    for file in os.listdir(directory):
        if file.endswith(".jpg"):
            jpg_files.append(os.path.join(directory, file))
    return jpg_files

def get_image_dimensions(image_path):
    with Image.open(image_path) as img:
        width, height = img.size
    return width, height

def label_to_mask(label, image_shape):
    """
    Convert a YOLOv8 segmentation label line to a binary mask.
    Expects label format: "class_index x1 y1 x2 y2 ..." (normalized)
    image_shape is (height, width)
    """
    parts = list(map(float, label.split()))
    # Ignore the first part (class index)
    coordinates = parts[1:]
    # Scale the coordinates to pixel values (note: image_shape=(height, width))
    x = np.array(coordinates[0::2]) * image_shape[1]
    y = np.array(coordinates[1::2]) * image_shape[0]
    # Clamp values within image boundaries
    x = np.clip(x, 0, image_shape[1]-1)
    y = np.clip(y, 0, image_shape[0]-1)
    # Get the polygon points (row then column order)
    rr, cc = polygon(y, x)
    mask = np.zeros(image_shape, dtype=np.uint8)
    mask[rr, cc] = 1
    return mask

def convert_to_binary_mask(mask):
    """
    Ensure that a mask is binary (0s and 1s).
    """
    return np.where(mask > 0, 1, 0)

def binary_mask_to_regionprops_dict(binary_mask):
    props = regionprops(binary_mask)
    selected_props = [
        'area', 'area_convex', 'major_axis_length', 'minor_axis_length', 'eccentricity',
        'equivalent_diameter', 'euler_number', 'extent', 'feret_diameter_max', 'perimeter', 'solidity'
    ]
    props_dicts = []
    for region in props:
        region_dict = {}
        for prop in selected_props:
            try:
                region_dict[prop] = getattr(region, prop)
            except AttributeError:
                region_dict[prop] = None
        props_dicts.append(region_dict)
    return props_dicts

# --- Ground Truth and SAM Everything Masks ---

def get_ground_truth(image_path, labels_dir):
    """
    Load ground truth masks from a label file.
    Assumes the label file is located in the provided labels_dir
    with the same base name as the image.
    """
    image_name = os.path.splitext(os.path.basename(image_path))[0]
    image_width, image_height = get_image_dimensions(image_path)
    print(f"Image dimensions for {image_name}: {image_width}x{image_height}")
    label_path = os.path.join(labels_dir, image_name + ".txt")
    masks = []
    if os.path.exists(label_path):
        with open(label_path, 'r') as file:
            lines = file.readlines()
            for line in lines:
                line = line.strip()
                if line:
                    # Note: image_shape is (height, width)
                    masks.append(label_to_mask(line, (image_height, image_width)))
    else:
        print(f"Label file not found for {image_name}")
    return masks

def get_sam_everything_masks(image_path, mask_generator):
    """
    Use SAM Everything mode to generate predicted masks.
    """
    image = Image.open(image_path).convert("RGB")
    image_np = np.array(image)
    masks_info = mask_generator.generate(image_np)
    # Convert the segmentation output to binary masks
    masks = [mask_info['segmentation'].astype(np.uint8)
             for mask_info in masks_info if mask_info.get('segmentation') is not None]
    # Ensure binary values (0 and 1)
    return [convert_to_binary_mask(mask) for mask in masks]

# --- Metrics Calculation (Per-Object Matching) ---

def mean_iou_precision_recall(gt_masks, pred_masks):
    """
    Calculate the mean Intersection over Union (IoU), precision, and recall between
    ground truth and predicted masks on a per-predicted-object basis.

    For each predicted mask, we compute its IoU with every ground truth mask,
    select the best (highest IoU) match, and then record the metrics.
    """
    iou_scores = []
    precision_scores = []
    recall_scores = []
    for pred_mask in pred_masks:
        # Ensure binary
        pred_mask = convert_to_binary_mask(pred_mask)
        pred_iou_scores = []
        pred_precision_scores = []
        pred_recall_scores = []
        for gt_mask in gt_masks:
            # Check that dimensions match
            if pred_mask.shape != gt_mask.shape:
                raise ValueError("Mismatched dimensions between ground truth and predicted masks.")
            # Compute intersection and union
            intersection = np.logical_and(gt_mask, pred_mask).sum()
            union = np.logical_or(gt_mask, pred_mask).sum()
            if union != 0 and intersection != 0:
                iou = intersection / union
                pred_iou_scores.append(iou)
                precision = intersection / pred_mask.sum()
                recall = intersection / gt_mask.sum()
                pred_precision_scores.append(precision)
                pred_recall_scores.append(recall)
        # If no valid match is found, score is zero
        if len(pred_iou_scores) == 0:
            iou_scores.append(0)
            precision_scores.append(0)
            recall_scores.append(0)
        else:
            # Select the best matching ground truth mask for this prediction
            best_iou = max(pred_iou_scores)
            index = pred_iou_scores.index(best_iou)
            iou_scores.append(pred_iou_scores[index])
            precision_scores.append(pred_precision_scores[index])
            recall_scores.append(pred_recall_scores[index])
    # Return the average across all predicted masks
    mean_iou_value = np.mean(iou_scores)
    mean_precision_value = np.mean(precision_scores)
    mean_recall_value = np.mean(recall_scores)
    return mean_iou_value, mean_precision_value, mean_recall_value


In [11]:
base_directory = "DualSight/powder/test/"
images_directory = os.path.join(base_directory, "images")
labels_directory = os.path.join(base_directory, "labels")

# Get list of JPG files
jpg_files = get_jpg_files(images_directory)
print("JPG Files in the directory:")
for f in jpg_files:
    print(f)

# SAM model parameters (update the path to your checkpoint)
sam_checkpoint = "models/sam_vit_l_0b3195.pth"
model_type = "vit_l"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load SAM and create an automatic mask generator for everything mode
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)
mask_generator = SamAutomaticMaskGenerator(sam)

# Lists to accumulate metrics
iou_list, prec_list, rec_list = [], [], []
results_file = "sam_everything_results.txt"

with open(results_file, 'a') as file:
    for image_path in jpg_files:
        print(f"Processing {os.path.basename(image_path)}")
        # Load ground truth masks from the labels directory
        gt_masks = get_ground_truth(image_path, labels_directory)
        if not gt_masks:
            print("No ground truth masks found; skipping this image.")
            continue

        # Get predicted masks from SAM Everything mode
        pred_masks = get_sam_everything_masks(image_path, mask_generator)
        if not pred_masks:
            print("No predicted masks generated; skipping this image.")
            continue

        # Calculate per-object metrics
        iou, precision, recall = mean_iou_precision_recall(gt_masks, pred_masks)
        iou_list.append(iou)
        prec_list.append(precision)
        rec_list.append(recall)

        # Write per-image metrics
        file.write(f"{os.path.basename(image_path)}: IoU={iou:.4f}, Precision={precision:.4f}, Recall={recall:.4f}\n")
        print(f"IoU: {iou:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}")

    # Compute overall averages
    avg_iou = np.mean(iou_list) if iou_list else 0
    avg_prec = np.mean(prec_list) if prec_list else 0
    avg_rec = np.mean(rec_list) if rec_list else 0
    summary = f"\nFinal Average Scores: IoU={avg_iou:.4f}, Precision={avg_prec:.4f}, Recall={avg_rec:.4f}\n"
    file.write(summary)
    print(summary)

JPG Files in the directory:
DualSight/powder/test/images/Cu-Ni-Powder_250x_10_SE_png.rf.cd93ec4589ad8f4e412cb1ec0e805016.jpg
DualSight/powder/test/images/HP743_5S_500x_png.rf.9ff406796462449f85c2039537f32d6f.jpg
DualSight/powder/test/images/S05_03_SE1_1000X65_png.rf.b1b275b60d329ab4fae1100a32dcfd4e.jpg
DualSight/powder/test/images/S02_02_SE1_300X18_png.rf.1a16e8c5f4e008cb2fc48c98b35778fb.jpg
DualSight/powder/test/images/RHA_00-45_500X07_png.rf.2e24ff0e093484de86e43a21ef7e62cb.jpg
DualSight/powder/test/images/S04_01_SE1_1000X45_png.rf.baa89207016e4da58f6ec0ab4f2b008f.jpg
DualSight/powder/test/images/S02_03_SE1_1000X24_png.rf.61ceee7fe0a4f4ccabd61c1e71524baf.jpg
DualSight/powder/test/images/Cu-Ni-Powder_250x_1_SE_V1_png.rf.675fe4943c221bb28cfe029af5482897.jpg
DualSight/powder/test/images/S01_03_SE1_500X11_png.rf.4b7fddca6f3354a728be814c4f62d5ae.jpg
DualSight/powder/test/images/MW_FeMnAl_500_22-11_png.rf.7095906a9fa6befd1eb267219b11b181.jpg
DualSight/powder/test/images/S02_01_SE1_1000X16_